# Analyze Garmin data prepocess it & export to blender for 3D vizualization

## Sources:
- [Analysis of Running Activities from Garmin Watch Using Python](https://towardsdatascience.com/analysis-of-runing-activities-from-garmin-watch-using-python-99609f83314e)

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path

# import 3rd-party modules
import pandas as pd
import numpy as np

# import local modules

## Define functions

In [145]:
# Create Function to explore the dataframes
def explore(df: pd.DataFrame) -> None:
    """
    Function to print general information about the dataframe
    """
    print("********** 1. General info of data **********")
    print(df.info())
    
    print("\n********** 2. Shape of data **********")
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {len(df.columns)}")
    
    print("\n********** 3. Number of missing values per column **********")
    print(df.isnull().sum())
    
    print("\n********** 4. Number of duplicated values **********")
    print(df.duplicated().sum())
    
    print("\n********** 5. Number of unique values per column (NaN non included) **********")
    print(df.nunique())
    
    print("\n********** 6. Statistical info of each column **********")
    # print(df.describe(include='all').T)
    return df.describe(include='all', datetime_is_numeric=True).T

def convert_strings_to_duration(str_series):
    """
    Function to convert series of strings to durations; workaround when multiple formats in series
    """
    return pd.to_timedelta(pd.to_datetime(str_series).dt.strftime("%H:%M:%S.%f"))

def convert_durations_to_minutes(durations_series):
    # durations_series.dt.hour*60 + durations_series.dt.minute + durations_series.dt.second/60
    return durations_series.dt.total_seconds()/60

## Read data

In [153]:
# set csv path
football_activities_df_path = Path("assets/data/garmin_data/football_1_year.csv")

# read csv into dataframe
football_activities_df = pd.read_csv(football_activities_df_path, parse_dates=True)

## Explore data (Exploratory data analysis)

In [10]:
football_activities_df.head()

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Min Temp,Surface Interval,Decompression,Best Lap Time,Number of Laps,Max Temp,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Other,2022-08-07 13:49:30,False,Grimbergen Football,8.90,"1,381",02:45:10,140,192,5.0,...,0.0,0:00,No,11:37.86.7,9,0.0,01:49:59,02:45:10,40,42
1,Other,2022-07-31 16:41:23,False,Grimbergen Football,0.62,375,00:30:13,152,185,3.3,...,0.0,0:00,No,30:13.30.7,1,0.0,00:13:03,00:30:13,40,42
2,Other,2022-07-31 14:20:23,False,Grimbergen Football,6.71,775,01:39:29,129,172,3.0,...,0.0,0:00,No,12:10.61.4,7,0.0,01:22:37,01:39:30,40,44
3,Other,2022-07-24 14:43:01,False,Grimbergen Football,4.96,854,01:32:10,137,180,3.4,...,0.0,0:00,No,11:03.70.5,5,0.0,01:07:09,01:49:56,40,45
4,Other,2022-07-17 13:53:02,False,Grimbergen Football,9.09,"1,479",02:22:50,150,195,5.0,...,0.0,0:00,No,04:52.04.1,10,0.0,01:52:40,02:23:16,40,45


In [11]:
explore(football_activities_df)

********** 1. General info of data **********
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Activity Type             40 non-null     object 
 1   Date                      40 non-null     object 
 2   Favorite                  40 non-null     bool   
 3   Title                     40 non-null     object 
 4   Distance                  40 non-null     float64
 5   Calories                  40 non-null     object 
 6   Time                      40 non-null     object 
 7   Avg HR                    40 non-null     int64  
 8   Max HR                    40 non-null     int64  
 9   Aerobic TE                40 non-null     object 
 10  Avg Run Cadence           40 non-null     object 
 11  Max Run Cadence           40 non-null     object 
 12  Avg Speed                 40 non-null     object 
 13  Max Speed            

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Activity Type,40,1,Other,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,40,40,2022-07-10 14:36:44,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Favorite,40,1,False,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,40,5,Grimbergen Football,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Distance,40.0,NaN,NaN,NaN,4.45575,3.036394,0.0,1.65,4.72,6.6275,10.14
Calories,40,40,260,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time,40,40,00:30:13,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Avg HR,40.0,NaN,NaN,NaN,139.125,13.291963,113.0,130.75,140.5,150.5,161.0
Max HR,40.0,NaN,NaN,NaN,173.65,20.041687,134.0,161.75,181.0,190.5,199.0
Aerobic TE,40,24,5.0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Clean data

In [154]:
# select columns to keep
selected_cols = [
    "Date",
    "Title",
    "Distance",
    "Calories",	"Time", "Avg HR", "Max HR", "Aerobic TE", "Avg Run Cadence",
    "Max Run Cadence", "Avg Speed", "Max Speed", "Total Ascent", "Total Descent", "Avg Stride Length", 
    "Best Lap Time", "Number of Laps", "Max Temp", "Moving Time", "Elapsed Time", "Min Elevation", "Max Elevation"
    ]

football_activities_df = football_activities_df[selected_cols]

### ToDo: find better solution to parse duration strings
ideas: 
- coerce error to convert error to nan and then fill na with other format
- consolidate strings in 1 format

In [155]:
# convert concerned cols to datetime & duration time
football_activities_df['Date'] = pd.to_datetime(football_activities_df['Date'])
football_activities_df['Time'] = convert_strings_to_duration(football_activities_df['Time'])
football_activities_df['Elapsed Time'] = convert_strings_to_duration(football_activities_df['Elapsed Time'])
football_activities_df['Best Lap Time'] = football_activities_df['Best Lap Time'].apply(lambda x: f"00:{x}" if x.count(":") == 1 else x) # need to add hh:
football_activities_df['Best Lap Time'] = pd.to_timedelta(football_activities_df['Best Lap Time'])
football_activities_df['Moving Time'] = convert_strings_to_duration(football_activities_df['Moving Time'])

# convert durations cols to number of minutes
duration_cols = football_activities_df.select_dtypes(include=["timedelta64[ns]"]).columns

for duration_col in duration_cols:
    football_activities_df[duration_col] = convert_durations_to_minutes(football_activities_df[duration_col])

In [161]:
# replace dummy value "--" by nan
football_activities_df.replace({'--':np.nan}, inplace=True)

# remove comma (to indicate thousands)
football_activities_df["Calories"] = football_activities_df["Calories"].str.replace(',', "")

In [178]:
# convert object cols to float
## select only object cols
object_cols = football_activities_df.select_dtypes(include='object').columns.to_list()

## remove title from object cols
object_cols.remove("Title")

## convert object cols to float
football_activities_df[object_cols] = football_activities_df[object_cols].astype("float")

## inspect cols info
football_activities_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               40 non-null     datetime64[ns]
 1   Title              40 non-null     object        
 2   Distance           40 non-null     float64       
 3   Calories           39 non-null     float64       
 4   Time               40 non-null     float64       
 5   Avg HR             40 non-null     int64         
 6   Max HR             40 non-null     int64         
 7   Aerobic TE         39 non-null     float64       
 8   Avg Run Cadence    39 non-null     float64       
 9   Max Run Cadence    39 non-null     float64       
 10  Avg Speed          39 non-null     float64       
 11  Max Speed          39 non-null     float64       
 12  Total Ascent       32 non-null     float64       
 13  Total Descent      31 non-null     float64       
 14  Avg Stride L

In [180]:
football_activities_df

,Date,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,Avg Run Cadence,Max Run Cadence,...,Total Ascent,Total Descent,Avg Stride Length,Best Lap Time,Number of Laps,Max Temp,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,2022-08-07 13:49:30,Grimbergen Football,8.90,1381.0,165.166667,140,192,5.0,50.0,232.0,...,1.0,1.0,1.02,11.631117,9,0.0,109.983333,165.166667,40.0,42.0
1,2022-07-31 16:41:23,Grimbergen Football,0.62,375.0,30.216667,152,185,3.3,27.0,232.0,...,NaN,NaN,0.75,30.221783,1,0.0,13.050000,30.216667,40.0,42.0
2,2022-07-31 14:20:23,Grimbergen Football,6.71,775.0,99.483333,129,172,3.0,64.0,248.0,...,4.0,5.0,1.07,12.176900,7,0.0,82.616667,99.500000,40.0,44.0
3,2022-07-24 14:43:01,Grimbergen Football,4.96,854.0,92.166667,137,180,3.4,54.0,232.0,...,9.0,9.0,0.98,11.061750,5,0.0,67.150000,109.933333,40.0,45.0
4,2022-07-17 13:53:02,Grimbergen Football,9.09,1479.0,142.833333,150,195,5.0,67.0,248.0,...,20.0,18.0,0.94,4.867350,10,0.0,112.666667,143.266667,40.0,45.0
5,2022-07-10 14:36:44,Grimbergen Football,3.92,566.0,60.550000,134,165,3.0,69.0,218.0,...,10.0,10.0,0.94,14.148567,4,0.0,50.800000,60.550000,40.0,45.0
6,2022-07-03 13:53:02,Grimbergen Football,7.04,1380.0,123.733333,154,193,5.0,54.0,237.0,...,1.0,0.0,1.01,4.826667,8,0.0,89.916667,123.816667,39.0,40.0
7,2022-06-12 15:00:00,Grimbergen Football,3.03,509.0,46.616667,155,198,4.2,65.0,224.0,...,7.0,10.0,1.00,0.061567,5,0.0,35.950000,46.616667,40.0,45.0
8,2022-06-12 13:55:41,Grimbergen Football,4.48,658.0,63.800000,145,192,3.9,65.0,220.0,...,7.0,7.0,1.07,9.890883,5,0.0,52.633333,63.800000,40.0,45.0
9,2022-06-09 12:23:53,Schaarbeek Football,2.39,358.0,40.033333,132,180,3.1,64.0,242.0,...,38.0,5.0,0.98,11.074117,3,0.0,30.133333,40.033333,76.0,109.0
